# EE 451: Communications Systems
## Lesson 17 - Baseband Digital & Synchronization

### Learning Objectives
By the end of this lesson, you will be able to:
- Analyze intersymbol interference (ISI) and its causes
- Interpret eye diagrams for signal quality assessment
- Apply Nyquist criterion for zero-ISI pulse shaping
- Design raised cosine filters for bandwidth-efficient transmission
- Explain carrier and symbol timing synchronization requirements
- Describe Phase-Locked Loop (PLL) and Costas loop operation for synchronization

### Textbook Reference
Haykin & Moher, Chapter 6

## Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
plt.rcParams['lines.linewidth'] = 2

print("Setup complete!")

## Part 1: Intersymbol Interference (ISI)

When digital pulses pass through a **band-limited channel**, they spread in time and overlap with adjacent symbols.

**Causes of ISI:**
1. Band-limited channel (finite bandwidth)
2. Multipath propagation (delayed copies)
3. Non-ideal filters

**Effect:** Sampling at symbol times picks up interference from neighboring symbols, increasing BER.

In [ ]:
# === ISI Demonstration: Rectangular Pulses Through Band-Limited Channel ===
np.random.seed(42)

R = 1e6              # 1 Msym/s
T_sym = 1 / R        # symbol period
fs = 50 * R          # 50x oversampling
sps = int(fs / R)    # samples per symbol

# Transmit 5 rectangular pulses
bits = np.array([1, 0, 1, 1, 0])
symbols = 2 * bits - 1  # +1, -1
rect_signal = np.repeat(symbols, sps)
t = np.arange(len(rect_signal)) / fs

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Clean rectangular pulses
axes[0, 0].plot(t * 1e6, rect_signal, linewidth=1.5, color='C0')
axes[0, 0].set_title('Transmitted Rectangular Pulses')
axes[0, 0].set_xlabel('Time (\u03bcs)')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].set_ylim([-1.8, 1.8])

# Different channel bandwidths
channel_bw = [5 * R, 2 * R, 0.8 * R]
titles = ['Wide Channel (5R\u209b)', 'Moderate Channel (2R\u209b)', 'Narrow Channel (0.8R\u209b)']

for idx, (bw, title) in enumerate(zip(channel_bw, titles)):
    b, a = signal.butter(5, bw / (fs / 2))
    filtered = signal.filtfilt(b, a, rect_signal)
    ax = axes.flatten()[idx + 1]
    ax.plot(t * 1e6, filtered, linewidth=1.5, color='C0')
    # Mark optimal sampling times
    sample_idx = np.arange(sps // 2, len(filtered), sps)
    ax.plot(t[sample_idx] * 1e6, filtered[sample_idx], 'ro', markersize=8,
            zorder=5, label='Sample points')
    ax.set_title(f'After {title}')
    ax.set_xlabel('Time (\u03bcs)')
    ax.set_ylabel('Amplitude')
    ax.set_ylim([-1.8, 1.8])
    ax.axhline(y=0, color='k', linestyle='-', alpha=0.3)
    ax.legend(fontsize=9)

plt.suptitle('ISI from Band-Limited Channel', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== ISI Observation ===")
print("Wide channel:     Pulses preserved, clean sampling")
print("Moderate channel: Some rounding, samples still distinguishable")
print("Narrow channel:   Severe ISI, samples corrupted by neighbors")

## Part 2: Nyquist Criterion for Zero ISI

**Nyquist's First Criterion:** For zero ISI, the pulse shape $p(t)$ must satisfy:

$$p(nT) = \begin{cases} 1 & n = 0 \\ 0 & n \neq 0 \end{cases}$$

**Ideal Nyquist pulse:** The sinc function
$$p(t) = \text{sinc}(t/T) = \frac{\sin(\pi t/T)}{\pi t/T}$$

- Zero crossings at every $t = nT$ (for $n \neq 0$)
- Minimum bandwidth: $B = 1/(2T) = R_s/2$
- **Problem:** Infinite duration, slow decay ($1/t$), sensitive to timing errors

In [ ]:
# === Nyquist Criterion: Sinc Pulse and ISI-Free Property ===

T = 1.0  # normalized symbol period
t_sinc = np.linspace(-5 * T, 5 * T, 2000)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Single sinc pulse with zero crossings ---
p = np.sinc(t_sinc / T)
axes[0].plot(t_sinc / T, p, 'C0-', linewidth=2)
for n in range(-5, 6):
    if n == 0:
        axes[0].plot(n, 1, 'ro', markersize=12, zorder=5)
    else:
        axes[0].plot(n, 0, 'ro', markersize=8, zorder=5)
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[0].set_xlabel('Time (t/T)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Sinc Pulse: Zero Crossings at nT')
axes[0].annotate('p(0) = 1', xy=(0, 1), xytext=(0.8, 0.9),
                 fontsize=11, arrowprops=dict(arrowstyle='->', color='red'),
                 color='red')
axes[0].annotate('p(nT) = 0 for n\u22600', xy=(1, 0), xytext=(2.0, -0.25),
                 fontsize=11, arrowprops=dict(arrowstyle='->', color='red'),
                 color='red')

# --- Multiple sinc pulses showing ISI-free summation ---
data_symbols = [1, -1, 1, 1, -1]
colors = ['C0', 'C1', 'C2', 'C3', 'C4']
composite = np.zeros_like(t_sinc)

for i, (sym, clr) in enumerate(zip(data_symbols, colors)):
    shifted = sym * np.sinc((t_sinc - (i - 2) * T) / T)
    axes[1].plot(t_sinc / T, shifted, color=clr, alpha=0.5, linewidth=1,
                 label=f'Symbol {i}: {sym:+d}')
    composite += shifted

axes[1].plot(t_sinc / T, composite, 'k-', linewidth=2.5, label='Composite')
for i, sym in enumerate(data_symbols):
    axes[1].plot(i - 2, sym, 'ko', markersize=10, zorder=5)
axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)
axes[1].set_xlabel('Time (t/T)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('ISI-Free: Sinc Pulses Sum Correctly at nT')
axes[1].legend(fontsize=8, loc='lower right')

plt.tight_layout()
plt.show()

print("=== Nyquist Criterion ===")
print("p(nT) = 1 for n=0, p(nT) = 0 for n \u2260 0")
print("Sinc pulse satisfies this \u2192 Zero ISI at sampling instants")
print(f"Minimum bandwidth: B = 1/(2T) = R\u209b/2")
print("\nProblem: Sinc has infinite duration and slow (1/t) decay")
print("\u2192 Need practical alternative: Raised Cosine filter")

## Part 3: Raised Cosine Filter

The **raised cosine** filter is the practical Nyquist pulse:

**Roll-off factor** $\alpha$ ($0 \leq \alpha \leq 1$):

$$BW = \frac{1 + \alpha}{2T} = \frac{(1 + \alpha) R_s}{2}$$

| $\alpha$ | Bandwidth | Time Decay | Practical Use |
|:--------:|:---------:|:----------:|:-------------:|
| 0 | $R_s/2$ (minimum) | Slowest ($1/t$) | Ideal only |
| 0.25 | $0.625 R_s$ | Moderate | DVB, LTE |
| 0.5 | $0.75 R_s$ | Faster ($1/t^3$) | Common |
| 1.0 | $R_s$ | Fastest | Maximum ISI margin |

In [ ]:
# === Raised Cosine Filter: Frequency Response and Impulse Response ===

T = 1.0  # normalized symbol period
f = np.linspace(-1.5 / T, 1.5 / T, 2000)
t_rc = np.linspace(-6 * T, 6 * T, 2000)

alphas = [0, 0.25, 0.5, 1.0]
colors = ['C0', 'C1', 'C2', 'C3']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Frequency response ---
for alpha, clr in zip(alphas, colors):
    H = np.zeros_like(f)
    for i, fi in enumerate(f):
        fa = abs(fi)
        if alpha == 0:
            H[i] = T if fa <= 1 / (2 * T) else 0
        else:
            f1 = (1 - alpha) / (2 * T)
            f2 = (1 + alpha) / (2 * T)
            if fa <= f1:
                H[i] = T
            elif fa <= f2:
                H[i] = T / 2 * (1 + np.cos(np.pi * T / alpha * (fa - f1)))
            else:
                H[i] = 0
    axes[0].plot(f * T, H / T, color=clr, linewidth=2,
                 label=f'\u03b1 = {alpha}')

axes[0].set_xlabel('Normalized Frequency (fT)')
axes[0].set_ylabel('|H(f)| / T')
axes[0].set_title('Raised Cosine Frequency Response')
axes[0].legend()
axes[0].set_xlim([-1.5, 1.5])

# --- Impulse response ---
for alpha, clr in zip(alphas, colors):
    if alpha == 0:
        h = np.sinc(t_rc / T)
    else:
        num = np.sinc(t_rc / T) * np.cos(np.pi * alpha * t_rc / T)
        denom = 1 - (2 * alpha * t_rc / T)**2
        denom[np.abs(denom) < 1e-10] = 1e-10
        h = num / denom
    axes[1].plot(t_rc / T, h, color=clr, linewidth=2,
                 label=f'\u03b1 = {alpha}')

# Mark symbol times
for n in range(-6, 7):
    axes[1].axvline(x=n, color='gray', linestyle=':', alpha=0.3)

axes[1].set_xlabel('Time (t/T)')
axes[1].set_ylabel('h(t)')
axes[1].set_title('Raised Cosine Impulse Response')
axes[1].legend()
axes[1].set_ylim([-0.4, 1.2])

plt.tight_layout()
plt.show()

# --- Bandwidth comparison ---
print("=== Raised Cosine Bandwidth ===")
print(f"{'Roll-off \u03b1':<12} {'BW (normalized)':<18} {'BW (1 Msym/s)':<18} {'Excess BW':<12}")
print("-" * 60)
R_ex = 1e6  # 1 Msym/s
for alpha in alphas:
    bw_norm = (1 + alpha) / 2
    bw_hz = bw_norm * R_ex
    excess = alpha * R_ex / 2
    print(f"{alpha:<12.2f} {bw_norm:<18.3f} {bw_hz/1e3:<14.0f} kHz  {excess/1e3:<12.0f} kHz")

## Part 4: Eye Diagram Generation

An **eye diagram** overlays many symbol periods to visualize signal quality:

| Metric | Meaning |
|--------|--------|
| **Eye height** | Noise margin (vertical opening) |
| **Eye width** | Timing margin (horizontal opening) |
| **Jitter** | Timing variations at zero crossings |

- **Open eye** = low ISI, good SNR, adequate timing margin
- **Closed eye** = high ISI, poor SNR, or severe jitter

In [ ]:
# === Eye Diagram: Noiseless vs Noisy ===
np.random.seed(42)

R = 1e6          # 1 Msym/s
fs = 20 * R      # 20x oversampling
sps = int(fs / R)
num_symbols = 300

# Random BPSK symbols
symbols = 2 * np.random.randint(0, 2, num_symbols) - 1

# Upsample
upsampled = np.zeros(num_symbols * sps)
upsampled[::sps] = symbols

# Raised cosine filter (alpha = 0.5)
alpha = 0.5
ntaps = 10 * sps + 1
t_filt = np.arange(-(ntaps // 2), ntaps // 2 + 1) / fs
h_rc = np.sinc(t_filt * R) * np.cos(np.pi * alpha * R * t_filt) / \
       (1 - (2 * alpha * R * t_filt)**2 + 1e-12)
h_rc /= np.sum(h_rc)

# Pulse-shape
tx = np.convolve(upsampled, h_rc, mode='same')

# Add noise (15 dB SNR)
snr_dB = 15
P_s = np.mean(tx**2)
P_n = P_s / 10**(snr_dB / 10)
rx = tx + np.sqrt(P_n) * np.random.randn(len(tx))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
traces_per_eye = 2 * sps  # 2 symbol periods wide

# Noiseless eye
for i in range(10 * sps, len(tx) - traces_per_eye, sps):
    trace = tx[i:i + traces_per_eye]
    t_trace = np.linspace(0, 2, len(trace))
    axes[0].plot(t_trace, trace, 'b-', alpha=0.08, linewidth=0.5)
axes[0].set_xlabel('Time (symbol periods)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title(f'Eye Diagram (Noiseless, \u03b1 = {alpha})')
axes[0].axhline(y=0, color='k', linestyle='-', alpha=0.3)

# Noisy eye
for i in range(10 * sps, len(rx) - traces_per_eye, sps):
    trace = rx[i:i + traces_per_eye]
    t_trace = np.linspace(0, 2, len(trace))
    axes[1].plot(t_trace, trace, 'b-', alpha=0.08, linewidth=0.5)
axes[1].set_xlabel('Time (symbol periods)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title(f'Eye Diagram (SNR = {snr_dB} dB, \u03b1 = {alpha})')
axes[1].axhline(y=0, color='k', linestyle='-', alpha=0.3)

plt.tight_layout()
plt.show()

print("=== Eye Diagram Interpretation ===")
print("Eye height:  Vertical opening at center \u2192 Noise margin")
print("Eye width:   Horizontal opening \u2192 Timing margin")
print("Clean open eye = low ISI + low noise")

## Part 5: Eye Diagram Dependence on SNR and Roll-off Factor

Two key parameters affect eye diagram quality:

- **SNR:** Higher SNR → cleaner eye, larger vertical opening
- **Roll-off $\alpha$:** Larger $\alpha$ → wider eye opening, but requires more bandwidth

In [ ]:
# === Eye Diagrams: Varying SNR (top row) and Roll-off (bottom row) ===
np.random.seed(42)

snr_values = [5, 10, 15, 20]
alpha_values = [0.1, 0.25, 0.5, 1.0]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))

# --- Row 1: Varying SNR (fixed alpha = 0.5) ---
for col, snr_val in enumerate(snr_values):
    P_n_var = P_s / 10**(snr_val / 10)
    rx_var = tx + np.sqrt(P_n_var) * np.random.randn(len(tx))
    for i in range(10 * sps, len(rx_var) - traces_per_eye, sps):
        trace = rx_var[i:i + traces_per_eye]
        t_trace = np.linspace(0, 2, len(trace))
        axes[0, col].plot(t_trace, trace, 'b-', alpha=0.06, linewidth=0.3)
    axes[0, col].set_title(f'SNR = {snr_val} dB')
    axes[0, col].set_ylim([-2.5, 2.5])
    axes[0, col].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    if col == 0:
        axes[0, col].set_ylabel('Amplitude')

# --- Row 2: Varying roll-off (fixed SNR = 15 dB) ---
snr_fixed = 15
for col, alpha_val in enumerate(alpha_values):
    # Build filter with this alpha
    h_var = np.sinc(t_filt * R) * np.cos(np.pi * alpha_val * R * t_filt) / \
            (1 - (2 * alpha_val * R * t_filt)**2 + 1e-12)
    h_var /= np.sum(h_var)
    tx_var = np.convolve(upsampled, h_var, mode='same')
    P_s_var = np.mean(tx_var**2)
    P_n_var = P_s_var / 10**(snr_fixed / 10)
    rx_var = tx_var + np.sqrt(P_n_var) * np.random.randn(len(tx_var))

    for i in range(10 * sps, len(rx_var) - traces_per_eye, sps):
        trace = rx_var[i:i + traces_per_eye]
        t_trace = np.linspace(0, 2, len(trace))
        axes[1, col].plot(t_trace, trace, 'b-', alpha=0.06, linewidth=0.3)
    bw_val = (1 + alpha_val) * R / (2e6)
    axes[1, col].set_title(f'\u03b1 = {alpha_val} (BW = {bw_val:.2f} MHz)')
    axes[1, col].set_ylim([-2.5, 2.5])
    axes[1, col].set_xlabel('Time (T)')
    axes[1, col].axhline(y=0, color='k', linestyle='-', alpha=0.3)
    if col == 0:
        axes[1, col].set_ylabel('Amplitude')

plt.suptitle('Eye Diagram Dependence on SNR and Roll-off Factor',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("=== Observations ===")
print("Row 1 (varying SNR):   Higher SNR \u2192 cleaner eye, larger opening")
print("Row 2 (varying \u03b1):    Larger \u03b1 \u2192 wider eye, but more bandwidth")
print("\nTrade-off: Bandwidth efficiency vs ISI/timing margin")

## Summary

### Key Formulas

| Concept | Formula |
|---------|--------|
| Minimum BW (zero ISI) | $B_{\min} = R_s / 2$ |
| Raised cosine BW | $B = (1 + \alpha) R_s / 2$ |
| Nyquist criterion | $p(nT) = \delta[n]$ |
| Ideal Nyquist pulse | $p(t) = \text{sinc}(t/T)$ |

### Key Takeaways

1. **ISI** arises when pulses spread through band-limited channels
2. **Nyquist criterion** defines conditions for zero ISI at sampling instants
3. **Raised cosine filter** is the practical solution ($\alpha$ trades BW for ISI margin)
4. **Eye diagrams** visualize signal quality: open eye = good, closed eye = bad
5. **Trade-off:** Larger $\alpha$ → wider eye opening but more bandwidth

### Practical Usage

| System | Typical $\alpha$ | Bandwidth |
|--------|:---------------:|:---------:|
| DVB-S2 | 0.20 – 0.35 | Near minimum |
| LTE | 0.22 | Spectral efficiency |
| WiFi | 0.25 | Moderate |
| Bluetooth | 0.50 | Relaxed |

### Next Topics
- Lesson 18: ISI mitigation, OFDM, spread spectrum introduction